# Wildfire Readiness Atlas — data preparation

This notebook creates the public state-level dashboard file from a current **official FEMA National Risk Index county CSV**. Attach that export as a Kaggle input dataset; do not add personal data, unpublished school data, or API keys.

The output is educational risk context only—not a live warning, fire prediction, or evacuation tool.

In [ ]:
from pathlib import Path
import csv, json
from collections import defaultdict
from datetime import date

# In Kaggle, change this only to the path of the official FEMA NRI county CSV you attached.
INPUT_CSV = Path('/kaggle/input/fema-national-risk-index/nri_counties.csv')
OUTPUT_JSON = Path('/kaggle/working/state-wildfire-risk.json')
SOURCE_VERSION = 'FEMA National Risk Index — enter the exact download version/date here'

assert INPUT_CSV.exists(), 'Attach the official FEMA NRI county CSV, then update INPUT_CSV.'

In [ ]:
REQUIRED = {'STATE', 'STATEABBRV', 'WFIR_EALT', 'WFIR_RISKS'}

def as_number(value):
    try:
        return float((value or '').replace(',', ''))
    except ValueError:
        return 0.0

with INPUT_CSV.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    missing = REQUIRED - set(reader.fieldnames or [])
    assert not missing, f'Missing required FEMA columns: {sorted(missing)}'
    groups = defaultdict(lambda: {'name':'', 'abbr':'', 'eal':0.0, 'score_sum':0.0, 'count':0})
    for row in reader:
        abbr = (row.get('STATEABBRV') or '').strip()
        if not abbr or abbr in {'PR', 'VI', 'GU', 'AS', 'MP'}:
            continue
        item = groups[abbr]
        item['name'] = (row.get('STATE') or abbr).title()
        item['abbr'] = abbr
        item['eal'] += as_number(row.get('WFIR_EALT'))
        item['score_sum'] += as_number(row.get('WFIR_RISKS'))
        item['count'] += 1

scores = [item['score_sum'] / item['count'] for item in groups.values() if item['count']]
low_cut, high_cut = sorted(scores)[round((len(scores)-1)*.33)], sorted(scores)[round((len(scores)-1)*.67)]

def risk_band(score):
    return 'High' if score >= high_cut else 'Moderate' if score >= low_cut else 'Low'

states = []
for item in groups.values():
    score = item['score_sum'] / item['count']
    states.append({'name':item['name'], 'abbr':item['abbr'], 'risk_band':risk_band(score),
                   'wildfire_risk_score':round(score, 3), 'wildfire_eal_total':f"${item['eal']:,.0f}",
                   'counties_included':item['count'],
                   'notes':'A state aggregation of county-level FEMA NRI wildfire fields; not a real-time forecast.'})

payload = {'metadata': {'status':'verified', 'title':'Wildfire Readiness Atlas', 'source':SOURCE_VERSION,
                        'updated_at':str(date.today()),
                        'disclaimer':'Educational use only; do not use for emergency decisions.'},
           'states': sorted(states, key=lambda state:state['name'])}
OUTPUT_JSON.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f'Wrote {len(states)} jurisdictions to {OUTPUT_JSON}')
assert len(states) >= 50, 'Expected at least 50 states; inspect the source export before publishing.'

## Before publishing

1. Record the FEMA file version and date in `SOURCE_VERSION`.
2. Review all jurisdictions; do not treat a missing record as low risk.
3. Download `state-wildfire-risk.json` from Kaggle Output and place it in `app/data/`.
4. Test the dashboard locally before uploading it to the VM.